In [2]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter

In [3]:
dataset = ["I love playing football on the weekends",
 "I enjoy hiking and camping in the mountains",
 "I like to read books and watch movies",
 "I prefer playing video games over sports",
 "I love listening to music and going to concerts"]

In [4]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(dataset)

In [5]:
k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)
# Predict the clusters for each document
y_pred = km.predict(X)
# Display the document and its predicted cluster in a table
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(dataset, y_pred)])
print(tabulate(table_data, headers="firstrow"))
# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
 print("Cluster %d:" % i)
 for ind in order_centroids[i, :10]:
  print(' %s' % terms[ind])
 print()

Document                                           Predicted Cluster
-----------------------------------------------  -------------------
I love playing football on the weekends                            0
I enjoy hiking and camping in the mountains                        1
I like to read books and watch movies                              1
I prefer playing video games over sports                           0
I love listening to music and going to concerts                    1

Top terms per cluster:
Cluster 0:
 playing
 on
 weekends
 football
 over
 sports
 video
 games
 prefer
 the

Cluster 1:
 to
 and
 read
 like
 books
 movies
 watch
 mountains
 in
 enjoy



/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [6]:
# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)


Purity: 0.6


In [10]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

# Download all required resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

# Setup
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

df = pd.DataFrame(dataset, columns=["Text"])
pd.set_option('display.max_colwidth', None)

# Step 1: Lowercase
def convert_to_lowercase(text):
    return text.lower()

df["lowercased"] = df["Text"].apply(convert_to_lowercase)
print("=== Lowercased ===")
print(df["lowercased"])

# Step 2: Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = []
    for word in words:
        if word.lower() not in stop_words:
            filtered_words.append(word)
    return " ".join(filtered_words)

df["stopwords_removed"] = df["lowercased"].apply(remove_stopwords)
print("\n=== Stopwords Removed ===")
print(df["stopwords_removed"])

#Step 3: Stemming 
stemmer = PorterStemmer()

def stem_text(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

df["stemmed"] = df["stopwords_removed"].apply(stem_text)
print("\n=== Stemmed ===")
print(df["stemmed"])

#Step 5: Lemmatization 
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_text(text):
    if not isinstance(text, str):
        return ""
    words = word_tokenize(text)
    pos_tags = pos_tag(words)
    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    return " ".join(lemmatized_words)

df["lemmatized"] = df["stopwords_removed"].apply(lemmatize_text)
print("\n=== Lemmatized ===")
print(df["lemmatized"])

cleaned_list = df["lemmatized"].tolist()
print(cleaned_list)


=== Lowercased ===
0            i love playing football on the weekends
1        i enjoy hiking and camping in the mountains
2              i like to read books and watch movies
3           i prefer playing video games over sports
4    i love listening to music and going to concerts
Name: lowercased, dtype: object

=== Stopwords Removed ===
0         love playing football weekends
1         enjoy hiking camping mountains
2           like read books watch movies
3      prefer playing video games sports
4    love listening music going concerts
Name: stopwords_removed, dtype: object

=== Stemmed ===
0       love play footbal weekend
1        enjoy hike camp mountain
2       like read book watch movi
3    prefer play video game sport
4    love listen music go concert
Name: stemmed, dtype: object

=== Lemmatized ===
0         love play football weekend
1        enjoy hike camping mountain
2         like read book watch movie
3       prefer play video game sport
4    love listening music go 

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/32d85ecf-770e-42c5-891c-
[nltk_data]     5ba4457b2ac3/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from collections import Counter

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_list)

k = 2 # Define the number of clusters
km = KMeans(n_clusters=k)
km.fit(X)
# Predict the clusters for each document
y_pred = km.predict(X)
# Display the document and its predicted cluster in a table
table_data = [["Document", "Predicted Cluster"]]
table_data.extend([[doc, cluster] for doc, cluster in zip(cleaned_list, y_pred)])
print(tabulate(table_data, headers="firstrow"))
# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()
for i in range(k):
 print("Cluster %d:" % i)
 for ind in order_centroids[i, :10]:
  print(' %s' % terms[ind])
 print()

# Calculate purity
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Document                           Predicted Cluster
-------------------------------  -------------------
love play football weekend                         0
enjoy hike camping mountain                        0
like read book watch movie                         1
prefer play video game sport                       0
love listening music go concert                    0

Top terms per cluster:
Cluster 0:
 play
 love
 football
 weekend
 mountain
 enjoy
 camping
 hike
 video
 game

Cluster 1:
 watch
 read
 book
 like
 movie
 sport
 video
 weekend
 play
 music

Purity: 0.8


Purity before pre-processing is 0.6, after pre-processing steps is 0.8